# TTNet – Colab الكامل (كل شيء في ملف واحد)  
**كل شيء على Google Drive • الكود من GitHub**

---
### خطوات التشغيل — **نوتبوك واحد لكل المراحل**
1. **Google Drive** — ربط حسابك (كل الملفات على Drive)
2. **GitHub** — استنساخ المشروع إلى مجلد المشروع
3. تثبيت المتطلبات وتوافق NumPy
4. تحميل الداتا واستخراج الفريمات
5. (اختياري) معاينة **مسار الفريمات** وخط الزمن قبل التدريب
6. التدريب (3 مراحل)
7. TensorBoard والاختبار وتحليل الأداء وDemo

> **GPU:** `Runtime → Change runtime type → T4 GPU` (أو أقوى)

> **المجلد على Drive (افتراضي):** `MyDrive/TTNet/` — غيّري الاسم في خلية الإعدادات إن رغبتِ.


## 0️⃣ تحقق من الـ GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('⚠️  لا يوجد GPU! تأكدي من Runtime → Change runtime type → T4 GPU')

## 1️⃣ ربط Google Drive أولاً
**شغّلي هذه الخلية قبل أي شيء يكتب على Drive.**

كل التدريب والداتا والـ checkpoints والـ logs والنتائج تُحفظ تحت مجلد واحد على حسابك (مثلاً `MyDrive/TTNet/`).

In [ ]:
from google.colab import drive
import os

# اسم المجلد داخل Google Drive (يمكن تغييره)
_DRIVE_PROJECT_NAME = 'TTNet'
PROJECT_DIR = f'/content/drive/MyDrive/{_DRIVE_PROJECT_NAME}'

if not os.path.isdir('/content/drive/MyDrive'):
    print('🔗 ارفعي الأذونات عند الطلب...')
    drive.mount('/content/drive')
    print('✅ تم ربط Google Drive')
else:
    print('✅ Google Drive مربوط مسبقاً')

os.makedirs(PROJECT_DIR, exist_ok=True)
print('\n📁 كل العمل سيُحفظ هنا (على حسابك):')
print('  ', PROJECT_DIR)
print('\n➡️ الشغلة التالية: شغّلي خلية «استنساخ من GitHub» أدناه.')

## 2️⃣ المشروع من GitHub + المسارات على Drive

- **`GITHUB_URL`**: ادخلي رابط مستودعك على GitHub (الافتراضي الريبو الرسمي أعلاه).
- الكود يُنسَخ إلى **`PROJECT_DIR/code`** على Drive؛ **dataset ، checkpoints ، logs ، results** كلها تحت **`PROJECT_DIR`**.

إذا كان الكود موجوداً مسبقاً على Drive، ستُطبع رسالة ولن يُعاد الاستنساخ.

In [ ]:
import os, subprocess
from pathlib import Path

assert os.path.isdir('/content/drive/MyDrive'), (
    'شغّلي خلية «ربط Google Drive» (الخطوة 1️⃣) أولاً!'
)

_DRIVE_PROJECT_NAME = 'TTNet'
TRAIN_GAME   = 'game_1'
TEST_GAME    = 'test_2'

GITHUB_URL = 'https://github.com/maudzung/TTNet-Realtime-for-Table-Tennis-Pytorch.git'
GIT_CLONE_EXTRA = ['--depth', '1']

PROJECT_DIR = f'/content/drive/MyDrive/{_DRIVE_PROJECT_NAME}'

VIDEO_DIR_DRIVE = os.path.join(PROJECT_DIR, 'videos')
os.makedirs(VIDEO_DIR_DRIVE, exist_ok=True)

DATASET_DIR     = os.path.join(PROJECT_DIR, 'dataset')
CODE_DIR        = os.path.join(PROJECT_DIR, 'code')
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
LOGS_DIR        = os.path.join(PROJECT_DIR, 'logs')
RESULTS_DIR     = os.path.join(PROJECT_DIR, 'results')
BASE_URL        = 'https://lab.osai.ai/datasets/openttgames/data/'

for d in (PROJECT_DIR, DATASET_DIR, CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)


def _find_src(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None


SRC_DIR = _find_src(CODE_DIR) if os.path.isdir(CODE_DIR) else None

if SRC_DIR:
    print(f'✅ الكود موجود مسبقاً على Drive:')
    print(f'    {SRC_DIR}')

elif GITHUB_URL:
    print('📥 git clone إلى:')
    print(f'    {CODE_DIR}')
    os.makedirs(CODE_DIR, exist_ok=True)
    if os.listdir(CODE_DIR):
        print('⚠️  مجلد code غير فارغ وليس فيه مشروع TTNet متوقّع.')
        print('    احذفي المجلد يدوياً من Drive أو أخليه فارغاً:', CODE_DIR)
        raise RuntimeError('تعارض مجلد code على Drive.')

    subprocess.run(['git', 'clone', *GIT_CLONE_EXTRA, GITHUB_URL, '.'],
                   cwd=CODE_DIR, check=True)
    SRC_DIR = _find_src(CODE_DIR)
    print(f'✅ تم استنساخ المشروع على Google Drive.')
    print(f'    SRC_DIR = {SRC_DIR}')

else:
    ZIP_PATH = os.path.join(PROJECT_DIR, 'TTNet.zip')
    if os.path.isfile(ZIP_PATH):
        subprocess.run(['unzip', '-q', '-o', ZIP_PATH, '-d', CODE_DIR], check=True)
        SRC_DIR = _find_src(CODE_DIR)
        print('✅ تم فتح ZIP:', SRC_DIR)
    else:
        raise RuntimeError(f'فعّلي GITHUB_URL أو ضعي TTNet.zip في {ZIP_PATH}')

ROOT_DIR = str(Path(SRC_DIR).parent) if SRC_DIR else CODE_DIR

print()
print('📋 كل الملفات أسفل هذا المسار (على حساب Google):')
print('   PROJECT_DIR  =', PROJECT_DIR)
print('   SRC_DIR      =', SRC_DIR or '❌')
print('   DATASET_DIR  =', DATASET_DIR)
print('   TRAIN_GAME   =', TRAIN_GAME, '| TEST_GAME =', TEST_GAME)

## 3️⃣ (اختياري) كود بدون GitHub — ZIP من Drive

إذا لا تستخدمين `git clone`: ارفقي `TTNet.zip` إلى **`MyDrive/TTNet/TTNet.zip`** ثم شغّلي الخلية أدناه (أو جهّني `GITHUB_URL = ''` في الخلية السابقة واستخدمي الـ ZIP).

In [ ]:
# اختياري: فقط إذا لم يعمل git clone وفيه TTNet.zip على Drive ضمن PROJECT_DIR

import os, subprocess
from pathlib import Path

assert 'PROJECT_DIR' in dir() and PROJECT_DIR, 'شغّلي خليتي الإعداد السابقة أولاً.'
ZIP_PATH = os.path.join(PROJECT_DIR, 'TTNet.zip')
CODE_DIR = os.path.join(PROJECT_DIR, 'code')

def _find_src_zip(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

SRC_NOW = _find_src_zip(CODE_DIR) if os.path.isdir(CODE_DIR) else None

if SRC_NOW:
    print('✅ الكود جاهز — لا حاجة لـ ZIP.')
elif os.path.isfile(ZIP_PATH):
    os.makedirs(CODE_DIR, exist_ok=True)
    print('📦 جاري فتح', ZIP_PATH, '...')
    subprocess.run(['unzip', '-q', '-o', ZIP_PATH, '-d', CODE_DIR], check=True)
    SRC_NOW = _find_src_zip(CODE_DIR)
    print('✅ تم — SRC_DIR:', SRC_NOW)
    if SRC_NOW:
        SRC_DIR = SRC_NOW
        ROOT_DIR = str(Path(SRC_NOW).parent)
else:
    print('ℹ️ لم يُعثر على TTNet.zip — إن استعملتِ git clone تجاهلي هذه الخلية.')

## 🔄 تحديث الكود من GitHub (git pull)
شغّلي هذه الخلية في أي وقت تريدين سحب آخر تعديلات من GitHub.

In [ ]:
import subprocess, os, re
from pathlib import Path

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip(), r.returncode

def _find_src(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

def _reapply_patches(src):
    """يعيد تطبيق التعديلات الضرورية بعد كل pull."""
    patched = []

    # ── Patch 0: إصلاح torch.load weights_only=False ─────────────────────────
    mu = Path(src) / 'models' / 'model_utils.py'
    mt = mu.read_text()
    mnt = re.sub(r'torch\.load\(([^,)]+),\s*map_location=([^,)]+)\)',
                 r'torch.load(\1, map_location=\2, weights_only=False)', mt)
    if mnt != mt:
        mu.write_text(mnt)
        patched.append('model_utils.py → weights_only=False')

    # ── Patch 1: تجاهل الأحداث ذات الصور المفقودة ────────────────────────────
    du = Path(src) / 'data_process' / 'ttnet_data_utils.py'
    txt = du.read_text()
    skip = ("                # Skip sequences where any of the 9 image files is missing\n"
            "                if not all(os.path.isfile(p) for p in img_path_list):\n"
            "                    continue\n\n")
    if skip not in txt:
        anchor = "                # Get segmentation path for the last frame in the sequence"
        du.write_text(txt.replace(anchor, skip + anchor))
        patched.append('ttnet_data_utils.py → skip missing images')

    # ── Patch 2: إصلاح np.int / np.float / np.bool / np.complex / np.object / np.str ──
    for py in Path(src).parent.rglob('*.py'):
        t = py.read_text(errors='ignore')
        nt = t
        # np.int
        nt = re.sub(r'dtype=np\.int\b', 'dtype=np.int32', nt)
        nt = re.sub(r'\.astype\(np\.int\)', '.astype(np.int32)', nt)
        nt = re.sub(r'np\.int\b(?!\d|e|_|8|1|3|6)', 'int', nt)
        # np.float
        nt = re.sub(r'dtype=np\.float\b', 'dtype=np.float32', nt)
        nt = re.sub(r'\.astype\(np\.float\)', '.astype(np.float32)', nt)
        nt = re.sub(r'np\.float\b(?!\d|_)', 'float', nt)
        # np.bool
        nt = re.sub(r'np\.bool\b(?!_|8)', 'bool', nt)
        # np.complex
        nt = re.sub(r'np\.complex\b(?!\d|_)', 'complex', nt)
        # np.object
        nt = re.sub(r'np\.object\b(?!_)', 'object', nt)
        # np.str
        nt = re.sub(r'np\.str\b(?!_)', 'str', nt)
        if nt != t:
            py.write_text(nt)
            patched.append(str(py.name))

    if patched:
        print('🔧 Patches أُعيد تطبيقها:')
        for p in patched: print(f'   ✅ {p}')
    else:
        print('✅ لا patches جديدة مطلوبة')

# ── ابحثي عن .git ──────────────────────────────────────────────────────────
git_dir = CODE_DIR
if not os.path.isdir(os.path.join(git_dir, '.git')):
    for sub in (os.listdir(CODE_DIR) if os.path.isdir(CODE_DIR) else []):
        c = os.path.join(CODE_DIR, sub)
        if os.path.isdir(os.path.join(c, '.git')):
            git_dir = c
            break

if not os.path.isdir(os.path.join(git_dir, '.git')):
    print('⚠️  لم يُعثر على git repo. استخدمي GITHUB_URL في خلية الإعدادات.')
else:
    print(f'📂 git repo: {git_dir}')

    # 1. stash التعديلات المحلية
    out, err, _ = _run(['git', 'stash'], cwd=git_dir)
    print(f'📦 stash: {out or "لا تعديلات محلية"}')

    # 2. git pull
    out, err, code = _run(['git', 'pull'], cwd=git_dir)
    if code == 0:
        print(f'⬇️  pull: {out or "Already up to date"}')
    else:
        print(f'⚠️  pull error: {err}')

    # 3. stash pop (إن وُجد)
    out2, err2, _ = _run(['git', 'stash', 'pop'], cwd=git_dir)
    if 'No stash' not in out2 and 'No stash' not in err2:
        print(f'📤 stash pop: {out2 or err2}')

    # 4. إعادة تطبيق الـ patches
    SRC_DIR  = _find_src(git_dir)
    ROOT_DIR = str(Path(SRC_DIR).parent) if SRC_DIR else git_dir
    if SRC_DIR:
        _reapply_patches(SRC_DIR)
        print(f'\n✅ الكود محدّث | SRC_DIR = {SRC_DIR}')
    else:
        print('⚠️  لم يُعثر على SRC_DIR')

## 3️⃣ تثبيت المتطلبات

In [ ]:
print('📦 تثبيت المكتبات المطلوبة...')

# المكتبات الأساسية (Colab لديه torch مثبت مسبقاً)
!pip install -q easydict==1.9 scikit-learn tqdm

# TurboJPEG – نثبّت الإصدار 1.x المتوافق مع libjpeg-turbo 2.x الموجود في Colab
!apt-get install -q -y libturbojpeg
!pip install -q "PyTurboJPEG==1.7.1"

# tensorboard
!pip install -q tensorboard

# تحقق من الإصدارات
import subprocess
result = subprocess.run(['python3', '-c',
    'from turbojpeg import TurboJPEG; tj = TurboJPEG(); print("✅ TurboJPEG يعمل بشكل صحيح")'],
    capture_output=True, text=True)
print(result.stdout or result.stderr)
print('✅ جميع المكتبات مثبتة')

## 4️⃣ إصلاح مشاكل التوافق مع NumPy الحديث
الكود الأصلي يستخدم `np.int` و`np.float` و`np.bool` وغيرها المحذوفة من NumPy 1.20+. نصلح هذا تلقائياً.

In [ ]:
import subprocess, os, re

# اجد المسار الصحيح للكود
# الكود قد يكون في مجلد فرعي داخل code/ حسب اسم ZIP
def find_src_dir(base):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files and 'config' in dirs:
            return root
    return None

SRC_DIR = find_src_dir(CODE_DIR)
if SRC_DIR is None:
    # fallback: try common patterns
    for sub in os.listdir(CODE_DIR):
        candidate = os.path.join(CODE_DIR, sub, 'src')
        if os.path.isdir(candidate):
            SRC_DIR = candidate
            break

print('📁 SRC_DIR:', SRC_DIR)
ROOT_DIR = os.path.dirname(SRC_DIR) if SRC_DIR else CODE_DIR

# إصلاح np.int / np.float / np.bool / np.complex / np.object / np.str → builtins
import re
from pathlib import Path

fixed = []
for py_file in Path(CODE_DIR).rglob('*.py'):
    text = py_file.read_text(errors='ignore')
    new_text = text
    # np.int
    new_text = re.sub(r'dtype=np\.int\b', 'dtype=np.int32', new_text)
    new_text = re.sub(r'\.astype\(np\.int\)', '.astype(np.int32)', new_text)
    new_text = re.sub(r'np\.int\b(?!\d|e|_|8|1|3|6)', 'int', new_text)
    # np.float
    new_text = re.sub(r'dtype=np\.float\b', 'dtype=np.float32', new_text)
    new_text = re.sub(r'\.astype\(np\.float\)', '.astype(np.float32)', new_text)
    new_text = re.sub(r'np\.float\b(?!\d|_)', 'float', new_text)
    # np.bool
    new_text = re.sub(r'np\.bool\b(?!_|8)', 'bool', new_text)
    # np.complex
    new_text = re.sub(r'np\.complex\b(?!\d|_)', 'complex', new_text)
    # np.object
    new_text = re.sub(r'np\.object\b(?!_)', 'object', new_text)
    # np.str
    new_text = re.sub(r'np\.str\b(?!_)', 'str', new_text)
    if new_text != text:
        py_file.write_text(new_text)
        fixed.append(str(py_file))

if fixed:
    print(f'✅ تم إصلاح {len(fixed)} ملفات:')
    for f in fixed:
        print('   ', f)
else:
    print('✅ لا توجد مشاكل توافق')

# ── إصلاح 2: تخطي الأحداث التي تنقصها صور (يتيح التدريب بأي عدد صور) ─────────
_data_utils = Path(SRC_DIR) / 'data_process' / 'ttnet_data_utils.py'
_du_text = _data_utils.read_text()
_skip_check = "                # Skip sequences where any of the 9 image files is missing\n                if not all(os.path.isfile(p) for p in img_path_list):\n                    continue\n\n"
if _skip_check not in _du_text:
    _old = "                # Get segmentation path for the last frame in the sequence"
    _du_text = _du_text.replace(_old, _skip_check + _old)
    _data_utils.write_text(_du_text)
    print('✅ تم إضافة فلتر الصور المفقودة في ttnet_data_utils.py')
else:
    print('✅ فلتر الصور المفقودة موجود مسبقاً')

## 5️⃣ تحميل الداتا (OpenTTGames) – فيديو واحد فقط

المصدر: [lab.osai.ai](https://lab.osai.ai/)

اختاري **فيديو تدريب واحد** و**فيديو اختبار واحد** من الجدول التالي:

| الاسم | الحجم | النوع |
|-------|-------|-------|
| `game_1` | 5.6 GB | تدريب |
| `game_2` | 10.8 GB | تدريب |
| `game_3` | 4.6 GB | تدريب |
| `game_4` | 3.9 GB | تدريب |
| `game_5` | 4.5 GB | تدريب |
| `test_2` | **0.2 GB** ✅ أصغر اختبار | اختبار |
| `test_3` | 0.5 GB | اختبار |
| `test_1` | 1.1 GB | اختبار |

> **نصيحة:** استخدمي `game_1` للتدريب و`test_2` للاختبار – هما الأنسب للكولاب المجاني.

In [ ]:
# المجلدات
for d in ['training/videos', 'training/annotations',
          'test/videos',     'test/annotations']:
    os.makedirs(f'{DATASET_DIR}/{d}', exist_ok=True)

def download_if_missing(url, dst, label):
    if os.path.isfile(dst):
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  [skip] {label} موجود مسبقاً ({size_mb:.0f} MB)')
        return
    print(f'  ⬇️  تحميل {label} ...')
    import subprocess
    subprocess.run(['wget', '-q', '--show-progress', url, '-O', dst], check=True)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  ✅ تم تحميل {label} ({size_mb:.0f} MB)')

# ── فيديو التدريب ─────────────────────────────────────────────────────────────
print(f'📥 فيديو التدريب: {TRAIN_GAME}')
download_if_missing(
    url = f'{BASE_URL}{TRAIN_GAME}.mp4',
    dst = f'{DATASET_DIR}/training/videos/{TRAIN_GAME}.mp4',
    label = f'{TRAIN_GAME}.mp4'
)
download_if_missing(
    url = f'{BASE_URL}{TRAIN_GAME}.zip',
    dst = f'{DATASET_DIR}/training/annotations/{TRAIN_GAME}.zip',
    label = f'{TRAIN_GAME}.zip (annotations)'
)

# ── فيديو الاختبار ────────────────────────────────────────────────────────────
print(f'📥 فيديو الاختبار: {TEST_GAME}')
download_if_missing(
    url = f'{BASE_URL}{TEST_GAME}.mp4',
    dst = f'{DATASET_DIR}/test/videos/{TEST_GAME}.mp4',
    label = f'{TEST_GAME}.mp4'
)
download_if_missing(
    url = f'{BASE_URL}{TEST_GAME}.zip',
    dst = f'{DATASET_DIR}/test/annotations/{TEST_GAME}.zip',
    label = f'{TEST_GAME}.zip (annotations)'
)

print(f'\n✅ اكتمل التحميل')
print(f'   تدريب: {TRAIN_GAME} | اختبار: {TEST_GAME}')

## 6️⃣ فك ضغط الـ Annotations

In [ ]:
import zipfile, shutil

def unzip_annotations(annos_dir):
    """
    فك ضغط كل ZIP في annos_dir داخل مجلد فرعي باسم اللعبة.
    game_1.zip → annotations/game_1/ball_markup.json ...
    """
    for zip_fn in os.listdir(annos_dir):
        if not zip_fn.endswith('.zip'):
            continue

        game_name  = zip_fn[:-4]                               # game_1
        zip_path   = os.path.join(annos_dir, zip_fn)
        out_folder = os.path.join(annos_dir, game_name)        # annotations/game_1/

        if os.path.isdir(out_folder) and os.listdir(out_folder):
            print(f'  [skip] {game_name}/ موجود مسبقاً')
            continue

        os.makedirs(out_folder, exist_ok=True)
        print(f'  📂 فك {zip_fn} → {game_name}/ ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_folder)

        # لو الملفات اتفكت مباشرةً في annos_dir بشكل خاطئ، ننقلها
        for item in ['ball_markup.json', 'events_markup.json', 'segmentation_masks']:
            src = os.path.join(annos_dir, item)
            dst = os.path.join(out_folder, item)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.move(src, dst)
                print(f'     نقل {item} → {game_name}/')

        # تحقق
        files = os.listdir(out_folder)
        print(f'  ✅ {game_name}/ يحتوي: {files}')

print('📂 فك ضغط annotations التدريب...')
unzip_annotations(f'{DATASET_DIR}/training/annotations')
print('📂 فك ضغط annotations الاختبار...')
unzip_annotations(f'{DATASET_DIR}/test/annotations')
print('✅ تم فك الضغط بالهيكل الصحيح')

## 6️⃣ب ضبط الـ Config على الفيديو الواحد المحمّل
الكود الأصلي يتوقع 5 فيديوهات تدريب + 7 اختبار. هنا نضبطه تلقائياً على الفيديو اللي حملناه.

In [ ]:
from pathlib import Path
import re

# مسار config.py
config_path = Path(SRC_DIR) / 'config' / 'config.py'
config_text = config_path.read_text()

# استبدال قوائم الألعاب المُشفّرة بالفيديو الواحد المحمّل
config_text = re.sub(
    r"configs\.train_game_list\s*=\s*\[.*?\]",
    f"configs.train_game_list = ['{TRAIN_GAME}']",
    config_text
)
config_text = re.sub(
    r"configs\.test_game_list\s*=\s*\[.*?\]",
    f"configs.test_game_list = ['{TEST_GAME}']",
    config_text
)

# عدد الفريمات المتتالية في التسلسل (فردي: 9، 15، 25…) — يجب أن يطابق استخراج الفريمات
NUM_FRAMES_SEQUENCE = 9
config_text = re.sub(
    r"configs\.num_frames_sequence\s*=\s*\d+",
    f"configs.num_frames_sequence = {NUM_FRAMES_SEQUENCE}",
    config_text,
)
_GLOBAL_NUM_FRAMES = NUM_FRAMES_SEQUENCE

config_path.write_text(config_text)

# تحقق
for line in config_text.splitlines():
    if 'game_list' in line or 'num_frames_sequence' in line:
        print(' ', line.strip())

print('\n✅ config.py تم ضبطه على:')
print(f'   train_game_list = [{TRAIN_GAME}]')
print(f'   test_game_list  = [{TEST_GAME}]')
print(f"   num_frames_sequence = {NUM_FRAMES_SEQUENCE}")


## 7️⃣ استخراج الفريمات
يُحذف أولاً مجلد الصور القديم **بالكامل** على الـ Colab وعلى Drive، ثم يُستخرج **كل فريم** الفيديو عبر `prepare_dataset/extract_all_images.py` (ليست مقتصرة على فريمات حول الأحداث).
**ملاحظة:** الحجم والوقت يكبران؛ تأكدي من مساحة Drive والـ GPU/CPU وقت طويل.

In [ ]:
import sys, glob as _glob, shutil, subprocess
PREPARE_DIR = os.path.join(ROOT_DIR, 'prepare_dataset')
LOCAL_DATASET = '/content/dataset_local'
sys.path.insert(0, PREPARE_DIR)

# NFS ما زال مستخدماً من خلية ضبط config — للتدريب فقط؛ الاستخراج هنا يأخذ كل الفريمات
nfs_info = int(globals().get('_GLOBAL_NUM_FRAMES', 9))
print(f'ℹ️  num_frames_sequence في التدريب = {nfs_info} (من الاستخراج هذا يُستخرج الفيديو كاملاً)')

def _ensure_video_annos_local():
    for split, game in [('training', TRAIN_GAME), ('test', TEST_GAME)]:
        dst_vid = os.path.join(LOCAL_DATASET, split, 'videos', f'{game}.mp4')
        dst_ann = os.path.join(LOCAL_DATASET, split, 'annotations', game)
        os.makedirs(os.path.dirname(dst_vid), exist_ok=True)
        os.makedirs(os.path.join(LOCAL_DATASET, split, 'images'), exist_ok=True)

        src_mp4 = os.path.join(DATASET_DIR, split, 'videos', f'{game}.mp4')
        if not os.path.isfile(src_mp4):
            raise FileNotFoundError(f'لا يوجد فيديو: {src_mp4}')
        if not os.path.isfile(dst_vid):
            print(f'📋 نسخ {game}.mp4 إلى local...')
            shutil.copy2(src_mp4, dst_vid)
            print('   ✅ تم')
        else:
            print(f'   [ok] {game}.mp4 موجود local')

        src_ann = os.path.join(DATASET_DIR, split, 'annotations', game)
        if not os.path.isdir(src_ann):
            raise FileNotFoundError(f'لا annotations: {src_ann}')
        if not os.path.isdir(dst_ann):
            shutil.copytree(src_ann, dst_ann)
            print(f'   ✅ annotations/{game}')
        else:
            print(f'   [ok] annotations/{game}')

def _purge_old_extracted_frames():
    """يحذف كل صور الاستخراج السابقة على local وعلى Drive."""
    paths = []
    for split, game in [('training', TRAIN_GAME), ('test', TEST_GAME)]:
        paths.append(os.path.join(LOCAL_DATASET, split, 'images', game))
        paths.append(os.path.join(DATASET_DIR, split, 'images', game))
    print('\n🗑️  حذف مجلدات الصور القديمة بالكامل (local + Drive)…')
    for folder in paths:
        if os.path.isdir(folder):
            n = len(os.listdir(folder))
            shutil.rmtree(folder)
            print(f'   🗑️  حُذف {folder} ({n} ملف كان فيه)')

print('خطوة 1: نسخ الفيديو والـ annotations')
_ensure_video_annos_local()
_purge_old_extracted_frames()

def _run_extract_all():
    cmd = [
        sys.executable, 'extract_all_images.py',
        '--dataset_dir', LOCAL_DATASET,
        '--dataset_types', 'training', 'test',
    ]
    print('\n🖼️  استخراج كل فريمات الفيديو (كامل المقطع، ليس حول الأحداث فقط)...')
    subprocess.run(cmd, cwd=PREPARE_DIR, check=True)

_run_extract_all()

print('\n📤 نسخ الصور إلى Google Drive...')
for split, game in [('training', TRAIN_GAME), ('test', TEST_GAME)]:
    src = os.path.join(LOCAL_DATASET, split, 'images', game)
    dst = os.path.join(DATASET_DIR, split, 'images', game)
    if os.path.isdir(src) and os.listdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'   ✅ {game}: {len(os.listdir(dst))} صورة → Drive')
    else:
        print(f'   ⚠️  {game}: فارغ!')

train_n = len(_glob.glob(os.path.join(LOCAL_DATASET, 'training', 'images', TRAIN_GAME, '*.jpg')))
test_n = len(_glob.glob(os.path.join(LOCAL_DATASET, 'test', 'images', TEST_GAME, '*.jpg')))
print(f'\n📊 بعد الاستخراج الكامل — {TRAIN_GAME}: {train_n} صورة | {TEST_GAME}: {test_n} صورة')


## 8️⃣ إعداد مجلدات الإخراج

## ✅ تحقق من الصور قبل التدريب — شغّلي هذه أولاً
**إذا طلع خطأ هنا لا تشغّلي التدريب.**

In [ ]:
import glob as _glob

LOCAL_DATASET = '/content/dataset_local'

train_local = _glob.glob(f'{LOCAL_DATASET}/training/images/{TRAIN_GAME}/*.jpg')
test_local  = _glob.glob(f'{LOCAL_DATASET}/test/images/{TEST_GAME}/*.jpg')
train_drive = _glob.glob(f'{DATASET_DIR}/training/images/{TRAIN_GAME}/*.jpg')

print('📊 عدد الصور المتاحة:')
print(f'   Local - {TRAIN_GAME} تدريب : {len(train_local)} صورة')
print(f'   Local - {TEST_GAME}  اختبار: {len(test_local)} صورة')
print(f'   Drive - {TRAIN_GAME} تدريب : {len(train_drive)} صورة')

# تحديد مصدر الداتا للتدريب
MIN_REQUIRED = 200   # الحد الأدنى للبدء بالتدريب

if len(train_local) >= MIN_REQUIRED:
    TRAIN_DATA_DIR = LOCAL_DATASET
    print(f'\n✅ سيتم التدريب من LOCAL: {LOCAL_DATASET}')
    print(f'   عدد الصور المتاحة للتدريب: {len(train_local)}')
elif len(train_drive) >= MIN_REQUIRED:
    TRAIN_DATA_DIR = DATASET_DIR
    print(f'\n⚠️  سيتم التدريب من Drive: {DATASET_DIR}')
else:
    TRAIN_DATA_DIR = None
    print(f'\n❌ الصور غير كافية (موجود: {len(train_local)}, مطلوب: {MIN_REQUIRED}+)')
    print('   شغّلي خلية 7️⃣ أولاً')
    raise SystemExit('⛔ شغّلي خلية 7 أولاً')

In [ ]:
print('📁 المجلدات:')
print('   Checkpoints:', CHECKPOINTS_DIR)
print('   Logs:       ', LOGS_DIR)
print('   Results:    ', RESULTS_DIR)

## 🎞️ مسار الفريمات (فهم التسلسل قبل التدريب)

كل عينة = **تسلسل فريمات** حول لحظة الحدث (bounce / net). **الفريم الأخير** في التسلسل هو الذي تُعلَّم عليه موضع الكرة والتجزئة؛ **الفريم المركزي** يقابل لحظة الحدث عند التوسّع التدريجي (smooth labelling).

الخلية التالية ترسم **خطاً زمنياً** و**معاينة** من عينة واحدة (بعد إكمال الداتا وضبط `config`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import sys
import os

assert SRC_DIR and os.path.isdir(SRC_DIR), 'شغّلي خلايا الكود والداتا أولاً'

_sys = sys.argv
sys.argv = ['nb', '--working-dir', PROJECT_DIR, '--no-val', '--smooth-labelling']
sys.path.insert(0, SRC_DIR)
from config.config import parse_configs
from data_process.ttnet_data_utils import train_val_data_separation

configs = parse_configs()
sys.argv = _sys

nf = int(getattr(configs, 'num_frames_sequence', 9))
half = (nf - 1) // 2
frames_rel = list(range(-half, half + 1))

fig, ax = plt.subplots(figsize=(14, 2.5))
ax.set_xlim(-half - 1, half + 2)
ax.set_ylim(-0.3, 1.0)
ax.axis('off')
for i, f in enumerate(frames_rel):
    if f == 0:
        c, lab = '#FF6B35', 'F0\nحدث'
    elif i == nf - 1:
        c, lab = '#E63946', f'F+{half}\nكرة+seg'
    elif f < 0:
        c, lab = '#A8DADC', f'{f}'
    else:
        c, lab = '#457B9D', f'+{f}'
    ax.add_patch(plt.Rectangle((f - 0.35, 0.2), 0.7, 0.55, color=c, ec='white', lw=1.2))
    ax.text(f, 0.47, lab, ha='center', va='center', fontsize=7, color='white', fontweight='bold')
ax.set_title(f'مسار الفريمات | num_frames_sequence = {nf}', fontsize=12, pad=12)
leg = [
    mpatches.Patch(color='#A8DADC', label='قبل الحدث'),
    mpatches.Patch(color='#FF6B35', label='F0 = الحدث'),
    mpatches.Patch(color='#457B9D', label='بعد الحدث'),
    mpatches.Patch(color='#E63946', label='آخر فريم = موضع الكرة'),
]
ax.legend(handles=leg, loc='upper center', bbox_to_anchor=(0.5, 1.35), ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

try:
    tr, va, _, _ = train_val_data_separation(configs)
    if not tr:
        print('لا عينات تدريب — أكملي الاستخراج.')
    else:
        img_paths, ball_pos, events, _seg = tr[0]
        ncols = min(nf, 5)
        nrows = (nf + ncols - 1) // ncols
        fig2, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.4 * nrows))
        axes = np.array(axes).ravel()
        for j, pth in enumerate(img_paths):
            im = cv2.cvtColor(cv2.imread(pth), cv2.COLOR_BGR2RGB)
            im = cv2.resize(im, (320, 128))
            axes[j].imshow(im)
            axes[j].axis('off')
            rel = j - half
            t = f'F{rel:+d}' if rel != 0 else 'F0'
            if j == nf - 1:
                t += '\\n(كرة)'
            axes[j].set_title(t, fontsize=8)
        for j in range(len(img_paths), len(axes)):
            axes[j].set_visible(False)
        fig2.suptitle('معاينة تسلسل فريمات (عينة أولى)', fontsize=11)
        plt.tight_layout()
        plt.show()
except Exception as e:
    print('تجاهلي أو أصلحي الداتا:', e)


## 9️⃣ التدريب – المرحلة الأولى
**Global Ball Detection + Segmentation فقط**

- `--no_local` `--no_event` → لا تدريب على Local أو Events
- `--global_weight 2` → تركيز على الكرة العالمية
- `--num_epochs 30`

In [ ]:
%cd "{SRC_DIR}"

# TRAIN_DATA_DIR تُحدد تلقائياً من خلية التحقق (local أسرع، Drive احتياطي)
!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase1 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-3 \
    --lr_type plateau \
    --no_local \
    --no_event \
    --global_weight 2 \
    --seg_weight 1 \
    --smooth-labelling \
    --earlystop_patience 5 \
    --checkpoint_freq 2 \
    --num_workers 2

print('✅ انتهى تدريب المرحلة الأولى')

## 🔟 التدريب – المرحلة الثانية
**Local Ball Detection + Event Spotting**

- نحمّل نتائج المرحلة الأولى (`pretrained_path`)
- `--overwrite_global_2_local` → ننسخ أوزان Global لـ Local
- نجمّد Global + Segmentation

In [ ]:
PHASE1_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase1/ttnet_phase1_best.pth'

if not os.path.isfile(PHASE1_CKPT):
    # ابحثي عن أحدث checkpoint
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase1'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE1_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Checkpoint المرحلة الأولى:', PHASE1_CKPT)

%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase2 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-3 \
    --lr_type plateau \
    --pretrained_path "{PHASE1_CKPT}" \
    --overwrite_global_2_local \
    --freeze_global \
    --freeze_seg \
    --local_weight 1 \
    --event_weight 1 \
    --smooth-labelling \
    --earlystop_patience 5 \
    --num_workers 2

print('✅ انتهى تدريب المرحلة الثانية')

## 1️⃣1️⃣ التدريب – المرحلة الثالثة (Fine-tuning الكل)
نفتح جميع الأوزان للتدريب معاً بـ learning rate منخفض.

In [ ]:
PHASE2_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase2/ttnet_phase2_best.pth'

if not os.path.isfile(PHASE2_CKPT):
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase2'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE2_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Checkpoint المرحلة الثانية:', PHASE2_CKPT)

%cd "{SRC_DIR}"

!python main.py \
    --working-dir "{PROJECT_DIR}" \
    --saved_fn ttnet_phase3 \
    --gpu_idx 0 \
    --batch_size 8 \
    --num_epochs 30 \
    --lr 1e-4 \
    --lr_type plateau \
    --pretrained_path "{PHASE2_CKPT}" \
    --global_weight 1 \
    --local_weight 1 \
    --event_weight 1 \
    --seg_weight 1 \
    --smooth-labelling \
    --earlystop_patience 7 \
    --num_workers 2

print('✅ انتهى التدريب الكامل')

## 1️⃣2️⃣ مراقبة التدريب بـ TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{LOGS_DIR}"

## 1️⃣3️⃣ اختبار الموديل (Test)

In [ ]:
PHASE3_CKPT = f'{CHECKPOINTS_DIR}/ttnet_phase3/ttnet_phase3_best.pth'

if not os.path.isfile(PHASE3_CKPT):
    ckpt_dir = f'{CHECKPOINTS_DIR}/ttnet_phase3'
    ckpts    = sorted([
        f for f in os.listdir(ckpt_dir) if f.endswith('.pth')
    ]) if os.path.isdir(ckpt_dir) else []
    PHASE3_CKPT = os.path.join(ckpt_dir, ckpts[-1]) if ckpts else ''

print('📌 Best Checkpoint:', PHASE3_CKPT)

%cd "{SRC_DIR}"

!python test.py \
    --working-dir "{PROJECT_DIR}" \
    --gpu_idx 0 \
    --pretrained_path "{PHASE3_CKPT}" \
    --save_test_output

## 1️⃣4️⃣ تشغيل تحليل أداء اللاعب

**رفعي فيديو اللاعب على Drive** ثم حددي المسار أدناه.

- `--assessment_video` : فيديو التقييم الأولي (10 كرات)
- `--stage_videos`     : فيديوهات المراحل
- `--starting_level`  : مستوى اللاعب (beginner/intermediate/advanced)

In [ ]:
# ── حدّدي مسارات الفيديوهات ───────────────────────────────────────────────────
PLAYER_VIDEO = f'{PROJECT_DIR}/videos/player_game.mp4'   # ← غيري هذا

# للتحقق من وجود الفيديو
if not os.path.isfile(PLAYER_VIDEO):
    print('⚠️  الفيديو غير موجود على المسار:', PLAYER_VIDEO)
    print('ارفعيه على Drive أو غيري المسار أعلاه')
else:
    print('✅ الفيديو موجود')

# ── تحميل الفيديو مباشرة من الجهاز (اختياري) ─────────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# PLAYER_VIDEO = list(uploaded.keys())[0]

In [ ]:
%cd "{SRC_DIR}"

# ── تشغيل نظام تحليل الأداء ──────────────────────────────────────────────────
!python performance_demo.py \
    --pretrained_path "{PHASE3_CKPT}" \
    --assessment_video "{PLAYER_VIDEO}" \
    --stage_videos "{PLAYER_VIDEO}" \
    --gpu_idx 0 \
    --max_retries 3 \
    --bounce_thresh 0.5 \
    --net_thresh 0.5

## 1️⃣5️⃣ تشغيل مباشر (Demo أصلي)
لمشاهدة تتبع الكرة والـ Segmentation على الفيديو.

In [ ]:
%cd "{SRC_DIR}"

DEMO_OUTPUT = f'{RESULTS_DIR}/demo'
os.makedirs(DEMO_OUTPUT, exist_ok=True)

!python demo.py \
    --working-dir "{PROJECT_DIR}" \
    --pretrained_path "{PHASE3_CKPT}" \
    --video_path "{PLAYER_VIDEO}" \
    --gpu_idx 0 \
    --output_format video \
    --save_demo_output

print('✅ Demo اكتمل. الفيديو محفوظ في:', DEMO_OUTPUT)

## 1️⃣6️⃣ تحميل النتائج
لتحميل الـ checkpoint وفيديو النتائج على جهازك.

In [ ]:
from google.colab import files

# ── تحميل Best Checkpoint ──────────────────────────────────────────────────────
if os.path.isfile(PHASE3_CKPT):
    print('⬇️  تحميل Best Checkpoint...')
    files.download(PHASE3_CKPT)

# ── تحميل فيديو Demo ──────────────────────────────────────────────────────────
demo_video = os.path.join(DEMO_OUTPUT, 'ttnet_phase3', 'result.mp4')
if os.path.isfile(demo_video):
    print('⬇️  تحميل فيديو Demo...')
    files.download(demo_video)
else:
    print('ℹ️  فيديو Demo غير موجود (ربما التدريب لم يكتمل بعد)')

---
## 📋 ملاحظات مهمة

| الموضوع | التفاصيل |
|---------|----------|
| **Google Drive** | الجذر: `MyDrive/TTNet/` — اسم المجلد من `_DRIVE_PROJECT_NAME` في الخليتين 1 و2 |
| **GitHub** | افتراضي: الريبو الرسمي؛ غيّري `GITHUB_URL` لفوركك. الكود في `{PROJECT_DIR}/code` |
| **وقت التدريب** | ~2-4 ساعات لكل مرحلة على T4 GPU |
| **حفظ الجلسة** | dataset + checkpoints + logs على مسار واحد تحت حساب Google |
| **استكمال التدريب** | استخدمي `--resume_path` بدل `--pretrained_path` |
| **Colab Pro** | يعطيك وقت جلسة أطول وـ A100 GPU أسرع |
| **الداتا** | ~10 GB كاملة، أو تقدري تشتغلي بـ game_1 فقط للاختبار |
| **Notebook واحد** | محتوى التدريب/التحليل الموسّع مُدمَج هنا؛ لا حاجة لملف ثانٍ |

### للاستكمال من checkpoint:
```python
# بدّلي --pretrained_path بـ --resume_path
!python main.py \
    --resume_path "{CHECKPOINTS_DIR}/ttnet_phase1/ttnet_phase1.pth" \
    ...
```
